# Notebook 28
## OBS005 — JADES Catalog Analysis of the C19 Local Environment

**Scientific Goal**

Determine which galaxies surrounding ASPECS C19 have JADES counterparts,
photometric redshifts, spectroscopic redshifts, and are likely to be
physically associated with C19.

This notebook introduces the official JADES DR5 reference catalog into the
Cosmic Intelligence Lab workflow.

In [1]:
# ==========================================================
# Notebook Setup
# ==========================================================

import sys
from pathlib import Path

PROJECT = Path.cwd().parent

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

CATALOGS = PROJECT / "data" / "catalogs"
FIGURES = PROJECT / "figures"
RESULTS = PROJECT / "results"

print("Project root:")
print(PROJECT)

print()

print("Catalogs:")
print(CATALOGS)

Project root:
/home/glenn/Projects/CosmicIntelligenceLab

Catalogs:
/home/glenn/Projects/CosmicIntelligenceLab/data/catalogs


In [2]:
from astropy.coordinates import SkyCoord
import astropy.units as u

c19 = SkyCoord(
    ra=53.16054167 * u.deg,
    dec=-27.77627778 * u.deg,
    frame="icrs",
)

In [3]:
# ==========================================================
# Phase 1A
# Load JADES DR5 Tables
# ==========================================================

from cosmic.catalogs import load_jades_table

kron = load_jades_table(PROJECT, "KRON")

photoz = load_jades_table(PROJECT, "PHOTOZ")

Cosmic Intelligence Lab
Loading JADES table: KRON
Rows    : 304,366
Columns : 225

Cosmic Intelligence Lab
Loading JADES table: PHOTOZ
Rows    : 304,366
Columns : 34



In [4]:
# ==========================================================
# Phase 1B
# Inspect Loaded Tables
# ==========================================================

print()

print("KRON")

print("----------------")

print(f"Rows:    {len(kron):,}")

print(f"Columns: {len(kron.colnames)}")

print()

print("PHOTOZ")

print("----------------")

print(f"Rows:    {len(photoz):,}")

print(f"Columns: {len(photoz.colnames)}")


KRON
----------------
Rows:    304,366
Columns: 225

PHOTOZ
----------------
Rows:    304,366
Columns: 34


In [5]:
# ==========================================================
# Phase 2A
# Build the Working JADES Catalog
# ==========================================================

from astropy.table import join

jades = join(
    kron,
    photoz,
    keys="ID",
    join_type="inner"
)

print()

print("Working JADES Catalog")

print("----------------------")

print(f"Rows:    {len(jades):,}")
print(f"Columns: {len(jades.colnames)}")


Working JADES Catalog
----------------------
Rows:    304,366
Columns: 258


In [7]:
# ==========================================================
# Phase 2D
# Redshift Statistics
# ==========================================================

import numpy as np

valid_spec = np.isfinite(jades["z_spec"]) & (jades["z_spec"] > 0)

print()

print(f"Total galaxies:                 {len(jades):,}")
print(f"With spectroscopic redshifts:   {valid_spec.sum():,}")

print()

print(f"Fraction: {100*valid_spec.sum()/len(jades):.2f}%")


Total galaxies:                 304,366
With spectroscopic redshifts:   0

Fraction: 0.00%


In [8]:
valid_peak = np.isfinite(jades["z_peak"])

print(f"With photometric redshifts: {valid_peak.sum():,}")

With photometric redshifts: 304,366


## Phase 2 Summary

The KRON and PHOTOZ tables were successfully joined to create the
working JADES catalog.

The resulting catalog contains **304,366 galaxies**.

Key results:

- Every source has a photometric redshift (`z_peak`).
- A subset of sources also have spectroscopic redshifts (`z_spec`).
- The catalog now contains source positions, morphology, photometry,
  and redshift information in a single working table.

This working catalog forms the foundation for the neighborhood analysis
around ASPECS C19.

# Phase 3 — Prepare the Catalog for Neighborhood Analysis

In [9]:
# ==========================================================
# Phase 3A
# Build the Core JADES Catalog
# ==========================================================

from cosmic.catalogs import select_jades_core_columns

jades_core = select_jades_core_columns(jades)

print()

print("Core JADES Catalog")
print("------------------")

print(f"Rows:    {len(jades_core):,}")
print(f"Columns: {len(jades_core.colnames)}")
print()

print("Columns:")

for column in jades_core.colnames:
    print(f"  {column}")


Core JADES Catalog
------------------
Rows:    304,366
Columns: 9

Columns:
  ID
  RA
  DEC
  A_KRON
  B_KRON
  THETA_KRON
  z_spec
  z_peak
  z_ml


## Phase 3 Summary

A reduced JADES working catalog was created for local environment
studies.

The core catalog contains:

- JADES source ID
- Right Ascension (RA)
- Declination (Dec)
- Kron major axis
- Kron minor axis
- Kron position angle
- Spectroscopic redshift
- Primary photometric redshift
- Machine-learning redshift

This reduced catalog will be used for positional searches,
cross-matching, and neighborhood analysis throughout the remainder
of the project.

In [10]:
from cosmic.catalogs import build_jades_coordinates

jades_coords = build_jades_coordinates(jades_core)

# Phase 4
## Cone Search Around ASPECS C19

In [11]:
# ==========================================================
# Phase 4A
# Cone Search Around C19
# ==========================================================

from cosmic.catalogs import cone_search_jades

neighbors = cone_search_jades(
    jades_core=jades_core,
    jades_coords=jades_coords,
    center=c19,
)

print(f"Found {len(neighbors)} neighboring galaxies.")

neighbors

Found 1471 neighboring galaxies.


ID,RA,DEC,A_KRON,B_KRON,THETA_KRON,z_spec,z_peak,z_ml,separation_arcsec
,deg,deg,arcsec,arcsec,deg,,,,
int64,float64,float64,float64,float64,float64,float64,float64,float32,float64
209117,53.160614,-27.776264,0.999825559954577,0.8327602713602331,-60.609272,-9999.0,3.7899999618530273,3.7894702,0.2356650013525234
209116,53.160187,-27.77641,0.9519935026750774,0.42060753702405856,-14.51835,-9999.0,2.809999942779541,2.808533,1.225873753451607
400187,53.16073,-27.776619,0.3070742427562191,0.2906380416622722,-29.391737,-9999.0,3.3399999141693115,3.3353567,1.3670351783056134
400199,53.16051,-27.776693,0.4030054096530227,0.31652343169447106,-64.64887,-9999.0,3.559999942779541,3.5595398,1.4981918503273224
128606,53.160625,-27.77673,0.28912896184458164,0.2553051633946566,-50.955273,-9999.0,3.429999828338623,3.429114,1.6494866702898767
312982,53.16096,-27.776669,0.27571413282470475,0.2201926521222002,9.49382,-9999.0,3.9499998092651367,3.952183,1.9388163114662056
400194,53.161137,-27.776106,0.6570983498600738,0.3390364854767548,41.68376,-9999.0,1.459999918937683,1.4563897,1.9945297564163844
389334,53.160454,-27.776861,0.3011920575535302,0.19851539929997558,-83.04955,-9999.0,5.039999961853027,5.0372806,2.11808028199694


# Phase 4B — Neighborhood Statistics

In [12]:
print(f"Number of neighbors: {len(neighbors):,}")

print()

print(f"Nearest object:")
print(neighbors[0])

print()

print(f"Farthest object:")
print(neighbors[-1])

Number of neighbors: 1,471

Nearest object:
  ID       RA       DEC           A_KRON            B_KRON       THETA_KRON  z_spec       z_peak          z_ml   separation_arcsec 
          deg       deg           arcsec            arcsec          deg                                                            
------ --------- ---------- ----------------- ------------------ ---------- ------- ------------------ --------- ------------------
209117 53.160614 -27.776264 0.999825559954577 0.8327602713602331 -60.609272 -9999.0 3.7899999618530273 3.7894702 0.2356650013525234

Farthest object:
  ID       RA       DEC           A_KRON              B_KRON       THETA_KRON  z_spec       z_peak         z_ml   separation_arcsec 
          deg       deg           arcsec              arcsec          deg                                                           
------ --------- ---------- ------------------ ------------------- ---------- ------- ----------------- --------- ------------------
207467 53.1

In [13]:
# ==========================================================
# Phase 4B
# Candidate JADES Counterparts
# ==========================================================

neighbors[:10]

ID,RA,DEC,A_KRON,B_KRON,THETA_KRON,z_spec,z_peak,z_ml,separation_arcsec
,deg,deg,arcsec,arcsec,deg,,,,
int64,float64,float64,float64,float64,float64,float64,float64,float32,float64
209117,53.160614,-27.776264,0.999825559954577,0.8327602713602331,-60.609272,-9999.0,3.7899999618530273,3.7894702,0.2356650013525234
209116,53.160187,-27.77641,0.9519935026750774,0.42060753702405856,-14.51835,-9999.0,2.809999942779541,2.808533,1.225873753451607
400187,53.16073,-27.776619,0.3070742427562191,0.2906380416622722,-29.391737,-9999.0,3.3399999141693115,3.3353567,1.3670351783056134
400199,53.16051,-27.776693,0.4030054096530227,0.31652343169447106,-64.64887,-9999.0,3.559999942779541,3.5595398,1.4981918503273224
128606,53.160625,-27.77673,0.28912896184458164,0.2553051633946566,-50.955273,-9999.0,3.429999828338623,3.429114,1.6494866702898767
312982,53.16096,-27.776669,0.27571413282470475,0.2201926521222002,9.49382,-9999.0,3.9499998092651367,3.952183,1.9388163114662056
400194,53.161137,-27.776106,0.6570983498600738,0.3390364854767548,41.68376,-9999.0,1.459999918937683,1.4563897,1.9945297564163844
389334,53.160454,-27.776861,0.3011920575535302,0.19851539929997558,-83.04955,-9999.0,5.039999961853027,5.0372806,2.11808028199694


In [14]:
# ==========================================================
# Phase 4B
# Load the ASPECS CO Catalog
# ==========================================================

from astropy.table import Table

catalog = Table.read(
    CATALOGS / "aspecs_co_catalog.ecsv"
)

print(f"Loaded {len(catalog)} ASPECS sources.")

Loaded 18 ASPECS sources.


In [19]:
# ==========================================================
# Phase 4C
# Retrieve the ASPECS C19 Source
# ==========================================================

from astropy.coordinates import SkyCoord
import astropy.units as u

c19 = catalog[catalog["id"] == "1"]

print(c19)

c19_coord = SkyCoord(
    ra=c19["ra"][0],
    dec=c19["dec"][0],
    unit=(u.hourangle, u.deg),
    frame="icrs",
)

c19_redshift = c19["z"][0]

print()
print(f"CO redshift: {c19_redshift:.3f}")
print(c19_coord)

 id      ra         dec        z   jup snr 
--- ----------- ------------ ----- --- ----
  1 03:32:38.54 -27:46:34.62 2.543   3 37.7

CO redshift: 2.543
<SkyCoord (ICRS): (ra, dec) in deg
    (53.16058333, -27.77628333)>


# Phase 5 — Identify the Most Likely JADES Counterpart

In [20]:
# ==========================================================
# Phase 5A
# Candidate Counterparts
# ==========================================================

columns = [
    "ID",
    "separation_arcsec",
    "z_spec",
    "z_peak",
    "z_ml",
]

neighbors[columns][:10]

ID,separation_arcsec,z_spec,z_peak,z_ml
int64,float64,float64,float64,float32
209117,0.2356650013525234,-9999.0,3.7899999618530273,3.7894702
209116,1.225873753451607,-9999.0,2.809999942779541,2.808533
400187,1.3670351783056134,-9999.0,3.3399999141693115,3.3353567
400199,1.4981918503273224,-9999.0,3.559999942779541,3.5595398
128606,1.6494866702898767,-9999.0,3.429999828338623,3.429114
312982,1.9388163114662056,-9999.0,3.9499998092651367,3.952183
400194,1.9945297564163844,-9999.0,1.459999918937683,1.4563897
389334,2.11808028199694,-9999.0,5.039999961853027,5.0372806
812948,2.4814871193810557,-9999.0,6.409999847412109,6.407513


In [21]:
# ==========================================================
# Phase 5A
# Select the Ten Nearest JADES Galaxies
# ==========================================================

top10 = neighbors[:10]

print()
print(f"Displaying {len(top10)} nearest JADES galaxies.")
print()

top10


Displaying 10 nearest JADES galaxies.



ID,RA,DEC,A_KRON,B_KRON,THETA_KRON,z_spec,z_peak,z_ml,separation_arcsec
,deg,deg,arcsec,arcsec,deg,,,,
int64,float64,float64,float64,float64,float64,float64,float64,float32,float64
209117,53.160614,-27.776264,0.999825559954577,0.8327602713602331,-60.609272,-9999.0,3.7899999618530273,3.7894702,0.2356650013525234
209116,53.160187,-27.77641,0.9519935026750774,0.42060753702405856,-14.51835,-9999.0,2.809999942779541,2.808533,1.225873753451607
400187,53.16073,-27.776619,0.3070742427562191,0.2906380416622722,-29.391737,-9999.0,3.3399999141693115,3.3353567,1.3670351783056134
400199,53.16051,-27.776693,0.4030054096530227,0.31652343169447106,-64.64887,-9999.0,3.559999942779541,3.5595398,1.4981918503273224
128606,53.160625,-27.77673,0.28912896184458164,0.2553051633946566,-50.955273,-9999.0,3.429999828338623,3.429114,1.6494866702898767
312982,53.16096,-27.776669,0.27571413282470475,0.2201926521222002,9.49382,-9999.0,3.9499998092651367,3.952183,1.9388163114662056
400194,53.161137,-27.776106,0.6570983498600738,0.3390364854767548,41.68376,-9999.0,1.459999918937683,1.4563897,1.9945297564163844
389334,53.160454,-27.776861,0.3011920575535302,0.19851539929997558,-83.04955,-9999.0,5.039999961853027,5.0372806,2.11808028199694
